# 🌾 Crop Recommendation — Tabular Machine Learning

Train a crop recommendation model from the uploaded **Crop recommendation dataset.csv**.

**Model:** CatBoost Classifier  
**Metrics:** Accuracy, Macro-F1, Weighted-F1, Top-3/Top-5 Accuracy, classification report, confusion matrix.

> `LOCATION` and `PREVIOUS_CROP` are not present in this dataset, so they are not fabricated.


In [ ]:
!pip -q install catboost scikit-learn pandas matplotlib seaborn joblib

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from catboost import CatBoostClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

DATA_PATH = "/Crop recommendation dataset.csv"
print("Dataset exists:", os.path.exists(DATA_PATH))


In [ ]:
df = pd.read_csv(DATA_PATH)

print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

print("\nMissing values:", df.isnull().sum().sum())
print("Duplicate rows:", df.duplicated().sum())

display(df.head())


In [ ]:
print("Number of crops:", df["CROPS"].nunique())
print("Number of soil types:", df["SOIL"].nunique())
print("Seasons:", df["SEASON"].unique())
print("Water sources:", df["WATER_SOURCE"].unique())

display(df["CROPS"].value_counts().to_frame("count"))


## Feature selection

`CROPS` is the target.

We exclude `SOWN`, `HARVESTED`, and crop-duration fields because they describe the crop itself and would not be clean farmer inputs at recommendation time. Location and previous crop are not present in the CSV.

The remaining fields are used as agricultural/environmental inputs.


In [ ]:
FEATURES = [
    "TYPE_OF_CROP",
    "SOIL",
    "SEASON",
    "WATER_SOURCE",
    "SOIL_PH",
    "SOIL_PH_HIGH",
    "TEMP",
    "MAX_TEMP",
    "WATERREQUIRED",
    "WATERREQUIRED_MAX",
    "RELATIVE_HUMIDITY",
    "RELATIVE_HUMIDITY_MAX",
    "N",
    "N_MAX",
    "P",
    "P_MAX",
    "K",
    "K_MAX"
]

TARGET = "CROPS"

missing = [c for c in FEATURES + [TARGET] if c not in df.columns]
if missing:
    raise ValueError(f"Missing columns: {missing}")

X = df[FEATURES].copy()
y = df[TARGET].copy()

CATEGORICAL_FEATURES = ["TYPE_OF_CROP", "SOIL", "SEASON", "WATER_SOURCE"]
cat_indices = [X.columns.get_loc(c) for c in CATEGORICAL_FEATURES]

print("X:", X.shape)
print("Classes:", y.nunique())
print("Categorical columns:", CATEGORICAL_FEATURES)


In [ ]:
# 70% train / 15% validation / 15% test
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, stratify=y, random_state=42
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, stratify=y_temp, random_state=42
)

print("Train:", X_train.shape)
print("Validation:", X_val.shape)
print("Test:", X_test.shape)


## Train CatBoost

CatBoost is suitable here because the dataset mixes numerical and categorical variables.


In [ ]:
model = CatBoostClassifier(
    iterations=1000,
    learning_rate=0.05,
    depth=8,
    loss_function="MultiClass",
    random_seed=42,
    verbose=100,
    allow_writing_files=False
)

model.fit(
    X_train,
    y_train,
    cat_features=cat_indices,
    eval_set=(X_val, y_val),
    use_best_model=True,
    early_stopping_rounds=80
)

print("✅ Training complete")


In [ ]:
y_pred = model.predict(X_test).ravel()

accuracy = accuracy_score(y_test, y_pred)
macro_f1 = f1_score(y_test, y_pred, average="macro")
weighted_f1 = f1_score(y_test, y_pred, average="weighted")

print(f"Test Accuracy:    {accuracy:.4f} ({accuracy*100:.2f}%)")
print(f"Test Macro-F1:    {macro_f1:.4f} ({macro_f1*100:.2f}%)")
print(f"Test Weighted-F1:{weighted_f1:.4f} ({weighted_f1*100:.2f}%)")


In [ ]:
probabilities = model.predict_proba(X_test)
class_names = model.classes_

def top_k_accuracy(y_true, probs, classes, k):
    top_indices = np.argsort(probs, axis=1)[:, -k:]
    top_labels = classes[top_indices]
    return np.mean([
        true in predicted
        for true, predicted in zip(y_true, top_labels)
    ])

for k in [1, 3, 5]:
    score = top_k_accuracy(y_test.to_numpy(), probabilities, class_names, k)
    print(f"Top-{k} Accuracy: {score:.4f} ({score*100:.2f}%)")


In [ ]:
print(classification_report(
    y_test, y_pred, digits=4, zero_division=0
))


In [ ]:
cm = confusion_matrix(y_test, y_pred, labels=class_names)

plt.figure(figsize=(20, 16))
sns.heatmap(
    cm,
    xticklabels=class_names,
    yticklabels=class_names,
    cmap="Blues",
    annot=False
)
plt.xlabel("Predicted")
plt.ylabel("True")
plt.title("Crop Recommendation Confusion Matrix")
plt.xticks(rotation=90)
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()


In [ ]:
importance = pd.Series(
    model.get_feature_importance(),
    index=FEATURES
).sort_values(ascending=False)

plt.figure(figsize=(10, 7))
importance.sort_values().plot(kind="barh")
plt.xlabel("Importance")
plt.title("CatBoost Feature Importance")
plt.tight_layout()
plt.show()

display(importance.to_frame("importance"))


## 🌾 Top-5 crop recommendation function

The returned percentage is the model's probability/confidence for each class, **not a guarantee of crop success**.


In [ ]:
def recommend_crops(input_data, top_k=5):
    row = pd.DataFrame([input_data])[FEATURES]

    probs = model.predict_proba(row)[0]
    indices = np.argsort(probs)[::-1][:top_k]

    return pd.DataFrame({
        "Rank": range(1, top_k + 1),
        "Crop": class_names[indices],
        "Model_Probability_%": np.round(probs[indices] * 100, 2)
    })

# Example — replace values with real farmer inputs.
example_input = {
    "TYPE_OF_CROP": "cereals",
    "SOIL": "Alluvial soil",
    "SEASON": "kharif",
    "WATER_SOURCE": "irrigated",
    "SOIL_PH": 6.5,
    "SOIL_PH_HIGH": 8.0,
    "TEMP": 27.0,
    "MAX_TEMP": 40.0,
    "WATERREQUIRED": 2000.0,
    "WATERREQUIRED_MAX": 2500.0,
    "RELATIVE_HUMIDITY": 70.0,
    "RELATIVE_HUMIDITY_MAX": 80.0,
    "N": 80.0,
    "N_MAX": 100.0,
    "P": 40.0,
    "P_MAX": 60.0,
    "K": 40.0,
    "K_MAX": 60.0
}

display(recommend_crops(example_input, top_k=5))


In [ ]:
# Save model + metadata for backend integration
MODEL_PATH = "/content/crop_recommendation_catboost.cbm"
META_PATH = "/content/crop_recommendation_metadata.pkl"

model.save_model(MODEL_PATH)

metadata = {
    "features": FEATURES,
    "categorical_features": CATEGORICAL_FEATURES,
    "target": TARGET,
    "class_names": list(class_names)
}
joblib.dump(metadata, META_PATH)

print("✅ Saved:", MODEL_PATH)
print("✅ Saved:", META_PATH)
!ls -lh /content/crop_recommendation_catboost.cbm /content/crop_recommendation_metadata.pkl
